In [1]:
!pip install -q openai pydantic pandas

In [16]:
from pydantic import BaseModel, Field
from typing import List, Optional

class InvoiceItem(BaseModel):
    name: str
    quantity: float = 1.0
    price: Optional[float] = None
    total: Optional[float] = None

class ExtractedInvoice(BaseModel):
    reasoning: str
    merchant_name: Optional[str] = None
    date: Optional[str] = None
    items: List[InvoiceItem] = Field(default_factory=list)
    total_amount: Optional[float] = None

print("[+] Pydantic-схему створено успішно.")

[+] Pydantic-схему створено успішно.


In [17]:
import json
from google.colab import files, userdata
from openai import OpenAI

uploaded = files.upload()

with open("ocr_result_improved.txt", "r", encoding="utf-8") as f:
    raw_ocr_text = f.read().strip()

with open("ner_extracted.json", "r", encoding="utf-8") as f:
    bert_entities = json.load(f)

client_groq = OpenAI(
    api_key=userdata.get("groq-api-key").strip(),
    base_url="https://api.groq.com/openai/v1"
)

print("[+] Groq API підключено.")
print("[+] OCR-текст завантажено.")
print("[+] NER-сутності завантажено.")
print("Кількість NER-сутностей:",
      len(bert_entities["entities"]))

Saving ner_extracted.json to ner_extracted (1).json
Saving ocr_result_improved.txt to ocr_result_improved (1).txt
[+] Groq API підключено.
[+] OCR-текст завантажено.
[+] NER-сутності завантажено.
Кількість NER-сутностей: 1


In [18]:
system_prompt = """
Ти — інтелектуальна система обробки касових чеків.

Проаналізуй OCR-текст та витягни:
- назву магазину;
- дату покупки у форматі YYYY-MM-DD;
- усі товари, їх кількість, ціну та загальну вартість;
- кінцеву суму до сплати.

Правила:
1. Не вигадуй інформацію, якої немає в тексті.
2. Не вважай готівку, решту та ПДВ товарами.
3. Якщо кількість не вказана, використовуй 1.
4. Розрізняй загальну суму товарів і суму до сплати.
5. Поверни тільки JSON.

Формат JSON:
{
  "reasoning": "Коротке пояснення",
  "merchant_name": "Назва магазину",
  "date": "YYYY-MM-DD",
  "items": [
    {
      "name": "Назва товару",
      "quantity": 1,
      "price": 0.0,
      "total": 0.0
    }
  ],
  "total_amount": 0.0
}
"""

response_temp1 = client_groq.chat.completions.create(
    model="qwen/qwen3.8-27b",
    temperature=1.0,
    max_tokens=900,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": raw_ocr_text
        }
    ]
)

temp1_json = response_temp1.choices[0].message.content

print("========== TEMPERATURE = 1.0 ==========")
print(temp1_json)

========== TEMPERATURE = 1.0 ==========
{
  "reasoning": "Витягнуто назву магазину з префіксту 'тзоВ ТВК \"Львівхолод\"'. Дата і час покупки знайдено в кінці чека (18.11.2021). Розпізнано 7 товарів з їхніми цінами та кількістю (конвертовано хескоди у кількість на основі суми). Готівка, решта, ПДВ та заокруглення ігноруються як товари. Загальна сума до сплати визначена як 'до сплатн: 281.00'.",
  "merchant_name": "ТОВ ТВК \"Львівхолод\"",
  "date": "2021-11-18",
  "items": [
    {
      "name": "ГОМІЛКА КУРЯЧА 1КГ",
      "quantity": 1,
      "price": 98.27,
      "total": 98.27
    },
    {
      "name": "КЕТЧУП ДО ШАШЛИКУ 400г",
      "quantity": 1,
      "price": 19.90,
      "total": 19.90
    },
    {
      "name": "ШАМІН МАРИН ЦІЛІ 420Г",
      "quantity": 1,
      "price": 51.24,
      "total": 51.24
    },
    {
      "name": "СИР ПЛ GOUDA 40% 7ог",
      "quantity": 3,
      "price": 9.50,
      "total": 28.50
    },
    {
      "name": "АНАНАСИ КУСОЧ 580г",
      "quantity": 1

In [19]:
few_shot_prompt = """
Ти — система структурованого розпізнавання касових чеків.

Правила:
1. Витягни всі реальні товарні позиції.
2. Рядки з кількістю та ціною належать наступному товару.
3. Не створюй окремий товар із числа, яке є ціною.
4. Для вагових товарів зберігай дробову кількість.
5. Не включай готівку, решту, ПДВ і заокруглення до товарів.
6. total_amount — значення з рядка "До сплати".
7. Не вигадуй відсутні товари.
8. Поверни тільки JSON.

Приклад:

OCR:
2.000
X
15.50
МОЛОКО 1Л
31.00
ХЛІБ
24.00
СУМА
55.00
ДО СПЛАТИ
55.00

JSON:
{
  "reasoning": "Знайдено 2 товари. Молоко: 2 × 15.50 = 31.00. Хліб: 1 × 24.00 = 24.00.",
  "merchant_name": null,
  "date": null,
  "items": [
    {
      "name": "МОЛОКО 1Л",
      "quantity": 2,
      "price": 15.50,
      "total": 31.00
    },
    {
      "name": "ХЛІБ",
      "quantity": 1,
      "price": 24.00,
      "total": 24.00
    }
  ],
  "total_amount": 55.00
}
"""

response_temp0 = client_groq.chat.completions.create(
    model="qwen/qwen3.8-27b",
    temperature=0.0,
    max_tokens=900,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": few_shot_prompt},
        {"role": "user", "content": raw_ocr_text}
    ]
)

temp0_json = response_temp0.choices[0].message.content

print("========== TEMPERATURE = 0.0 + FEW-SHOT ==========")
print(temp0_json)

temp0_data = ExtractedInvoice.model_validate_json(temp0_json)

print("\n[+] JSON успішно перевірено Pydantic.")
print("Кількість товарів:", len(temp0_data.items))
print("До сплати:", temp0_data.total_amount)

========== TEMPERATURE = 0.0 + FEW-SHOT ==========
{
  "reasoning": "Виявлено 5 товарних позицій. 1) Гомілка куряча: 1.312 кг × 74.90 = 98.27. 2) Кетчуп до шашлику: 1 × 19.90 = 19.90. 3) Шампіньон маринований: 1 × 51.24 = 51.24. 4) Сир Gouda: 3.000 × 9.50 = 28.50. 5) Анонаси: 3.000 × 10.44 = 31.32. Загальна сума до сплати 281.00.",
  "merchant_name": "МАГАЗИН",
  "date": "18.11.2021",
  "items": [
    {
      "name": "ГОМІЛКА КУРЯЧА 1КГ",
      "quantity": 1.312,
      "price": 74.90,
      "total": 98.27
    },
    {
      "name": "КЕТЧУП ДО ШАШЛИКУ 400г",
      "quantity": 1,
      "price": 19.90,
      "total": 19.90
    },
    {
      "name": "ШАМПІН МАРИН ЦІЛІ 420Г",
      "quantity": 1,
      "price": 51.24,
      "total": 51.24
    },
    {
      "name": "СИР GOUDA 40% 70г",
      "quantity": 3,
      "price": 9.50,
      "total": 28.50
    },
    {
      "name": "АНАНАСИ КУСОЧ 580г",
      "quantity": 3,
      "price": 10.44,
      "total": 31.32
    }
  ],
  "total_amount": 28

In [20]:
from decimal import Decimal

def validate_invoice(invoice):
    items_sum = sum(
        Decimal(str(item.total))
        for item in invoice.items
        if item.total is not None
    )

    payable = Decimal(str(invoice.total_amount))
    difference = abs(payable - items_sum)

    is_valid = difference <= Decimal("0.05")

    print("========== ПЕРЕВІРКА ЧЕКА ==========")
    print("Кількість товарів:", len(invoice.items))
    print("Сума товарів:", items_sum, "грн")
    print("До сплати:", payable, "грн")
    print("Різниця:", difference, "грн")

    if is_valid:
        print("VALIDATION PASSED")
    else:
        print("VALIDATION FAILED")
        print("Потрібна перевірка оператором.")

    return is_valid

validation_result = validate_invoice(temp0_data)

========== ПЕРЕВІРКА ЧЕКА ==========
Кількість товарів: 5
Сума товарів: 229.23 грн
До сплати: 281.0 грн
Різниця: 51.77 грн
VALIDATION FAILED
Потрібна перевірка оператором.


In [21]:
import pandas as pd

ner_list = bert_entities.get("entities", [])

bert_persons = [
    e["word"] for e in ner_list
    if e.get("entity_group") == "PER"
]

bert_orgs = [
    e["word"] for e in ner_list
    if e.get("entity_group") == "ORG"
]

bert_dates = [
    e["word"] for e in ner_list
    if e.get("entity_group") == "DATE"
]

df_comparison = pd.DataFrame({
    "Показник": [
        "Назва магазину",
        "Дата покупки",
        "Кількість товарів",
        "Сума до сплати",
        "Перевірка арифметики",
        "Розпізнані особи"
    ],
    "BERT (NER)": [
        ", ".join(bert_orgs) or "Не знайдено",
        ", ".join(bert_dates) or "Не знайдено",
        "Не визначає",
        "Не визначає",
        "Не виконує",
        ", ".join(bert_persons) or "Не знайдено"
    ],
    "LLM (Qwen)": [
        temp0_data.merchant_name,
        temp0_data.date,
        len(temp0_data.items),
        temp0_data.total_amount,
        "FAILED" if not validation_result else "PASSED",
        "Не виділялися окремо"
    ]
})

display(df_comparison)

df_comparison.to_csv(
    "bert_vs_llm_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("[+] Порівняльну таблицю збережено.")

,Показник,BERT (NER),LLM (Qwen)
0,Назва магазину,Не знайдено,МАГАЗИН
1,Дата покупки,Не знайдено,18.11.2021
2,Кількість товарів,Не визначає,5
3,Сума до сплати,Не визначає,281.0
4,Перевірка арифметики,Не виконує,FAILED
5,Розпізнані особи,Гуненюк Валентина Михайлівна Чек,Не виділялися окремо


[+] Порівняльну таблицю збережено.


In [22]:
import json

final_result = temp0_data.model_dump()

final_result["validation"] = {
    "status": "PASSED" if validation_result else "FAILED",
    "calculated_sum": round(
        sum(item.total or 0 for item in temp0_data.items), 2
    ),
    "total_amount": temp0_data.total_amount,
    "requires_human_review": not validation_result
}

with open("invoice_result.json", "w", encoding="utf-8") as f:
    json.dump(final_result, f, ensure_ascii=False, indent=4)

print("========== ФІНАЛЬНИЙ JSON ==========")
print(json.dumps(final_result, ensure_ascii=False, indent=4))

print("\n[+] Результат збережено у invoice_result.json")

========== ФІНАЛЬНИЙ JSON ==========
{
    "reasoning": "Виявлено 5 товарних позицій. 1) Гомілка куряча: 1.312 кг × 74.90 = 98.27. 2) Кетчуп до шашлику: 1 × 19.90 = 19.90. 3) Шампіньон маринований: 1 × 51.24 = 51.24. 4) Сир Gouda: 3.000 × 9.50 = 28.50. 5) Анонаси: 3.000 × 10.44 = 31.32. Загальна сума до сплати 281.00.",
    "merchant_name": "МАГАЗИН",
    "date": "18.11.2021",
    "items": [
        {
            "name": "ГОМІЛКА КУРЯЧА 1КГ",
            "quantity": 1.312,
            "price": 74.9,
            "total": 98.27
        },
        {
            "name": "КЕТЧУП ДО ШАШЛИКУ 400г",
            "quantity": 1.0,
            "price": 19.9,
            "total": 19.9
        },
        {
            "name": "ШАМПІН МАРИН ЦІЛІ 420Г",
            "quantity": 1.0,
            "price": 51.24,
            "total": 51.24
        },
        {
            "name": "СИР GOUDA 40% 70г",
            "quantity": 3.0,
            "price": 9.5,
            "total": 28.5
        },
        {
    